# 10a Prepare the SI Concept Corpus

Join screened pair-year PMID membership to the existing abstract cache and SI output. This notebook creates the fixed development/confirmation pair split and estimates labeling volume before any model calls.

The split is deterministic (20% confirmation, seed 42). Articles shared by development and confirmation pairs are marked as shared; they are excluded from concept-generation examples in notebook 10b.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipeline_config import cfg, cfg_path, load_pipeline_config
from amr.pipelines.concepts import build_si_concept_paths, load_concept_corpus, corpus_preflight
from amr.si.concepts import (
    SEED_CONCEPTS, concept_catalog_hash, estimate_classification_cost,
    classify_concept_abstracts, build_yearly_concept_signals,
    rolling_origin_concept_forecasts, summarise_incremental_forecast_gains,
    propose_concept_children,
)

PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)
CONCEPT_PATHS = build_si_concept_paths(PROJECT_ROOT)
MODEL_NAME = str(cfg(PIPELINE_CONFIG, "llm.model", "gpt-5-nano"))

YEARLY_SCREENED_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_screened", PROJECT_ROOT)
ABSTRACT_CACHE_PATH = cfg_path(PIPELINE_CONFIG, "paths.abstract_cache", PROJECT_ROOT)
SI_PATH = PROJECT_ROOT / "outputs" / "parquets" / "yearly_si.parquet"
HELDOUT_FRACTION = 0.20
SPLIT_SEED = 42


## Load and save the corpus

In [ ]:
corpus, pair_split = load_concept_corpus(
    screened_path=YEARLY_SCREENED_PATH,
    abstracts_path=ABSTRACT_CACHE_PATH,
    si_path=SI_PATH,
    heldout_fraction=HELDOUT_FRACTION,
    split_seed=SPLIT_SEED,
)
CONCEPT_PATHS["directory"].mkdir(parents=True, exist_ok=True)
unique_abstracts = corpus[["pmid", "title", "abstract", "abstract_available"]].drop_duplicates("pmid")
corpus.drop(columns=["title", "abstract"]).to_parquet(CONCEPT_PATHS["corpus"], index=False)
unique_abstracts.to_parquet(CONCEPT_PATHS["abstracts"], index=False)
pair_split.to_csv(CONCEPT_PATHS["pair_split"], index=False)
preflight = corpus_preflight(corpus)
preflight

## Classification cost preflight

One request classifies the full current catalog for one unique abstract. Update token assumptions from a small measured pilot before enabling a full run.

In [ ]:
MODEL_NAME = str(cfg(PIPELINE_CONFIG, "llm.model", "gpt-5-nano"))
PRICING = cfg(PIPELINE_CONFIG, "cost_estimation.pricing_by_model", {}).get(MODEL_NAME, {})
INPUT_TOKENS_PER_CALL = 1600
OUTPUT_TOKENS_PER_CALL = 180
cost_estimate = estimate_classification_cost(
    preflight["unique_abstracts_to_classify"],
    estimated_input_tokens_per_call=INPUT_TOKENS_PER_CALL,
    estimated_output_tokens_per_call=OUTPUT_TOKENS_PER_CALL,
    input_usd_per_million_tokens=float(PRICING.get("input_usd_per_1m_tokens", 0.0)),
    output_usd_per_million_tokens=float(PRICING.get("output_usd_per_1m_tokens", 0.0)),
)
print({"model": MODEL_NAME, **cost_estimate})
if not PRICING:
    print("No configured price found for this model; update cost_estimation.pricing_by_model before estimating spend.")


## Data checks

In [ ]:
assert corpus[["pathogen", "antibiotic", "year", "pmid"]].duplicated().sum() == 0
assert set(pair_split["split"]) == {"development", "confirmation"}
assert pair_split.groupby("split")[["pathogen", "antibiotic"]].size().min() > 0
corpus.groupby("split").agg(
    pairs=("pathogen", "nunique"),
    pair_year_article_links=("pmid", "size"),
    unique_pmids=("pmid", "nunique"),
    abstract_coverage=("abstract_available", "mean"),
)